**ENTRENAMIENTO DE CLASIFICADOR DE EMOCIONES (HOG + PCA + KNN)**

En este cuaderno se explora un clasificador basado en vecinos más cercanos, combinando la extracción de gradientes orientados, estandarización, reducción de dimensionalidad con **PCA** y ponderación por distancia inversa.

El algoritmo KNN carece de una fase de optimización explícita, trasladando la carga computacional a la fase de inferencia al requerir el cálculo de distancias sobre el espacio de referencia. La reducción con PCA resulta crítica para mitigar la maldición de la dimensionalidad y acelerar la búsqueda de los vecinos más próximos.

**IMPORTS NECESARIOS Y DEPENDENCIAS**

Se importan los algoritmos de scikit-learn correspondientes a KNN, PCA, escalado estándar y serialización, así como las utilidades compartidas.

In [9]:
import time
import joblib
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from comun import TAM, Extractor, cargar_dataset, informe, registrar_entrenamiento

**CONFIGURACIÓN DE RUTAS Y ARCHIVO DE SALIDA**

Se especifican las rutas de lectura para entrenamiento, test y salida.

In [10]:
RUTA_TRAIN = "dataset/train"
RUTA_TEST = "dataset/test"
SALIDA = "modelos/hog_pca_knn.joblib"

**CARGA Y NORMALIZACIÓN DEL CONJUNTO DE DATOS**

Se leen los ejemplos de cada categoría en escala de grises, normalizando todas las imágenes al tamaño uniforme de 64x64 píxeles.

In [11]:
imgs_tr, ytr = cargar_dataset(RUTA_TRAIN, tam=TAM)
imgs_te, yte = cargar_dataset(RUTA_TEST, tam=TAM)

dataset/train: 30241 imágenes | por clase: {'angry': 3995, 'duckface': 1968, 'fear': 4097, 'happy': 7215, 'neutral': 4965, 'sad': 4830, 'surprise': 3171}
dataset/test: 7396 imágenes | por clase: {'angry': 958, 'duckface': 329, 'fear': 1024, 'happy': 1774, 'neutral': 1233, 'sad': 1247, 'surprise': 831}


**EXTRACCIÓN DE CARACTERÍSTICAS HOG**

Se calculan los vectores de histogramas de gradientes orientados para todos los rostros del conjunto.

In [12]:
extractor = Extractor("hog")
Xtr = extractor.transform(imgs_tr)
Xte = extractor.transform(imgs_te)

**DEFINICIÓN DEL PIPELINE CLASIFICADOR**

Se construye la canalización encadenando la estandarización `StandardScaler`, la reducción a 100 componentes principales con `PCA` y el clasificador `KNeighborsClassifier` con vecinos ponderados por el inverso de la distancia euclídea.

In [13]:
clasificador = make_pipeline(
    StandardScaler(),
    PCA(n_components=100, random_state=42),
    KNeighborsClassifier(n_neighbors=7, weights="distance"),
)

**AJUSTE DEL MODELO KNN**

Se indexan los datos de entrenamiento y se proyectan sobre el espacio de componentes principales, midiendo el tiempo de construcción del índice de vecinos.

In [14]:
t0 = time.time()
clasificador.fit(Xtr, ytr)
t_entreno = time.time() - t0

**EVALUACIÓN CUANTITATIVA SOBRE EL CONJUNTO DE TEST**

Se calcula la clase más votada por los $k$ vecinos más cercanos para cada muestra de test, reportando el informe de clasificación y los tiempos de respuesta por imagen.

In [15]:
t0 = time.time()
pred = clasificador.predict(Xte)
t_pred = time.time() - t0
metricas = informe(yte, pred, t_entreno, t_pred)

              precision    recall  f1-score   support

       angry       0.45      0.33      0.38       958
    duckface       0.65      0.79      0.72       329
        fear       0.45      0.33      0.38      1024
       happy       0.54      0.77      0.64      1774
     neutral       0.39      0.45      0.42      1233
         sad       0.45      0.27      0.33      1247
    surprise       0.60      0.64      0.62       831

    accuracy                           0.50      7396
   macro avg       0.50      0.51      0.50      7396
weighted avg       0.49      0.50      0.48      7396

Matriz de confusión (filas = real, columnas = predicho):
[[ 312   24   95  204  172   98   53]
 [   4  261   10   24   16    9    5]
 [  82   15  335  194  175  111  112]
 [  67   31   55 1358  138   75   50]
 [  92   22   71  313  559  100   76]
 [ 110   36  116  297  289  334   65]
 [  26   11   64  104   70   23  533]]

Accuracy test: 0.4992 | F1 macro: 0.4976
Tiempo de entrenamiento: 1.8 s
Infere

**SERIALIZACIÓN Y GUARDADO DEL PIPELINE COMPLETO**

Se empaqueta la secuencia completa de extracción HOG y clasificación KNN en un único objeto `Pipeline`.

In [16]:
modelo = Pipeline([("extractor", extractor), ("clasificador", clasificador)])
joblib.dump(modelo, SALIDA)
print("Modelo guardado en", SALIDA)
registrar_entrenamiento(SALIDA, t_entreno, ms_imagen=1000 * t_pred / len(yte), **metricas)

Modelo guardado en modelos/hog_pca_knn.joblib
Registrado en informe_entrenamiento\registro_modelos.json
